## Phase 0: Setup & Dataset

Loading the Bitext Customer Support dataset (~27,000 labeled tickets) via Hugging Face's `datasets` library, and inspecting its structure — including the `flags` column, which tags linguistic variations (colloquial, typos, negation, etc.) used to generate diverse phrasings per intent.

In [2]:
!pip install datasets lime scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
from datasets import load_dataset

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = dataset['train'].to_pandas()

print(df.shape)
df.head()

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

(26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [4]:
print(df.columns.tolist())
print()
print(df['intent'].value_counts())
print()
print(df['category'].value_counts())

['flags', 'instruction', 'category', 'intent', 'response']

intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent          999
check_payment_methods        999
delivery_period              999
newsletter_subscription      999
get_invoice                  999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
set_up_shipping_address      997
check_refund_policy          997
create_account               997
get_refund                   997
review                       997
delivery_options             995
delete_account               995
recover_password             995
track_order                  995
change_shipping_address      973
check_cancellation_fee       950
Name: count, dtype: int64

category
ACCOUN

In [5]:
df[['instruction', 'response', 'intent', 'category']].sample(5)

,instruction,response,intent,category
20863,I do not know how to reset my user profile PIN...,Definitely! I understand that resetting your u...,recover_password,ACCOUNT
8256,ya have a free number to talk with customer se...,Thank you for contacting! I certainly recogniz...,contact_customer_service,CONTACT
10701,how can I create a {{Account Type}} account fo...,Thank you for your interest in creating a {{Ac...,create_account,ACCOUNT
7983,help me to check what hours i can call custome...,Thank you for reaching out! I'm here to help y...,contact_customer_service,CONTACT
23490,there is an ixsue setting my shipping address up,I apologize for the inconvenience you're exper...,set_up_shipping_address,SHIPPING


## Phase 1: Ticket Classification

Cleaning ticket text and training a Logistic Regression classifier (with TF-IDF features) to predict ticket intent. TF-IDF + Logistic Regression was chosen over BERT specifically because it pairs cleanly with LIME for explainability — TF-IDF features are literal words, so explanations map directly onto human-readable terms. After discovering near-duplicate rows in the dataset, they are deduplicated before splitting to avoid train/test leakage.

In [6]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\{\{.*?\}\}', '', text)   # remove placeholders like {{Order Number}}
    text = re.sub(r'[^a-z\s]', '', text)       # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()   # collapse extra whitespace
    return text

df['clean_text'] = df['instruction'].apply(clean_text)
df[['instruction', 'clean_text']].sample(5)

,instruction,clean_text
22601,do you have an address to submita review for y...,do you have an address to submita review for y...
11113,information about the cancellation of my {{Acc...,information about the cancellation of my account
21177,I do not know what to do to report an error wi...,i do not know what to do to report an error wi...
20865,want assistance to retrieve my profile pin,want assistance to retrieve my profile pin
26453,I'm waiting for a reimbursement of {{Currency ...,im waiting for a reimbursement of


In [8]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['intent']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=3000)
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

In [11]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.9929302325581395

                          precision    recall  f1-score   support

            cancel_order       0.99      0.99      0.99       200
            change_order       0.96      0.99      0.98       199
 change_shipping_address       0.99      0.99      0.99       195
  check_cancellation_fee       1.00      1.00      1.00       190
           check_invoice       0.98      0.99      0.99       200
   check_payment_methods       1.00      1.00      1.00       200
     check_refund_policy       1.00      0.99      1.00       199
               complaint       1.00      1.00      1.00       200
contact_customer_service       1.00      0.99      1.00       200
     contact_human_agent       0.99      0.99      0.99       200
          create_account       0.99      0.98      0.98       199
          delete_account       0.98      1.00      0.99       199
        delivery_options       0.98      1.00      0.99       199
         delivery_period       1.00      0.99

In [12]:
print("Duplicate clean_text rows:", df['clean_text'].duplicated().sum())
print("Total rows:", len(df))

Duplicate clean_text rows: 3394
Total rows: 26872


In [13]:
df_dedup = df.drop_duplicates(subset='clean_text').reset_index(drop=True)
print("Rows before:", len(df))
print("Rows after dedup:", len(df_dedup))

Rows before: 26872
Rows after dedup: 23478


In [14]:
X = df_dedup['clean_text']
y = df_dedup['intent']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=3000)
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.9885008517887564

                          precision    recall  f1-score   support

            cancel_order       1.00      0.86      0.93        81
            change_order       0.89      0.98      0.93       166
 change_shipping_address       1.00      0.98      0.99       192
  check_cancellation_fee       0.99      1.00      0.99       185
           check_invoice       0.98      0.98      0.98       170
   check_payment_methods       1.00      1.00      1.00       197
     check_refund_policy       1.00      1.00      1.00       194
               complaint       1.00      1.00      1.00       199
contact_customer_service       1.00      0.99      1.00       194
     contact_human_agent       0.99      1.00      0.99       193
          create_account       0.98      0.98      0.98       160
          delete_account       0.98      0.99      0.99       169
        delivery_options       0.99      0.95      0.97       126
         delivery_period       0.99      1.00

## Phase 2: Explainability with LIME

Using LIME to explain individual classifier predictions by showing which words drove each decision.

In [15]:
!pip install lime -q

from lime.lime_text import LimeTextExplainer

def predict_proba(texts):
    vectorized = vectorizer.transform(texts)
    return model.predict_proba(vectorized)

class_names = model.classes_
explainer = LimeTextExplainer(class_names=class_names)

In [16]:
cancel_order_test_indices = y_test[y_test == 'cancel_order'].index
cancel_order_positions = [X_test_raw.index.get_loc(i) for i in cancel_order_test_indices[:8]]

for pos in cancel_order_positions:
    text = X_test_raw.iloc[pos]
    pred = model.predict(vectorizer.transform([text]))[0]
    print(f"[{pos}] True: cancel_order | Predicted: {pred} | Text: {text}")

[15] True: cancel_order | Predicted: cancel_order | Text: help me cancel purchase
[154] True: cancel_order | Predicted: cancel_order | Text: have problems wth cancelling order
[291] True: cancel_order | Predicted: cancel_order | Text: need help to cancel purchase
[379] True: cancel_order | Predicted: change_order | Text: question about cancellung purchase
[452] True: cancel_order | Predicted: cancel_order | Text: i need assistance cancelling purcjase
[483] True: cancel_order | Predicted: change_order | Text: is it possible to cxancel order
[503] True: cancel_order | Predicted: cancel_order | Text: i would like to cancel purchase i need help
[617] True: cancel_order | Predicted: cancel_order | Text: can you help me canceling order


In [17]:
sample_indices = [0, 15, 483]

for idx in sample_indices:
    text = X_test_raw.iloc[idx]
    true_label = y_test.iloc[idx]
    pred_label = model.predict(vectorizer.transform([text]))[0]
    pred_class_idx = list(class_names).index(pred_label)

    exp = explainer.explain_instance(text, predict_proba, num_features=8, labels=[pred_class_idx])
    exp.save_to_file(f'lime_explanation_{idx}.html')

    print(f"--- Ticket {idx} ---")
    print(f"Text: {text}")
    print(f"True label: {true_label} | Predicted: {pred_label}")
    print()

--- Ticket 0 ---
Text: information about editing my shipping address
True label: change_shipping_address | Predicted: change_shipping_address

--- Ticket 15 ---
Text: help me cancel purchase
True label: cancel_order | Predicted: cancel_order

--- Ticket 483 ---
Text: is it possible to cxancel order
True label: cancel_order | Predicted: change_order



## Phase 3: Retrieval (RAG) for Resolutions

Using a sentence-transformer (BERT-based, fine-tuned for semantic similarity) to embed both the knowledge base of past resolutions and incoming tickets, then retrieving the most similar past resolution via cosine similarity. Knowledge base embeddings are computed once, upfront; each incoming ticket is embedded fresh at query time.

In [18]:
!pip install sentence-transformers -q

from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [19]:
knowledge_base = df_dedup.drop_duplicates(subset='response').reset_index(drop=True)
print("Knowledge base size:", len(knowledge_base))

kb_embeddings = embedder.encode(knowledge_base['response'].tolist(), show_progress_bar=True, convert_to_tensor=True)

Knowledge base size: 23478


Batches:   0%|          | 0/734 [00:00<?, ?it/s]

In [20]:
def retrieve_best_resolution(ticket_text, top_k=1):
    ticket_embedding = embedder.encode(ticket_text, convert_to_tensor=True)

    similarities = util.cos_sim(ticket_embedding, kb_embeddings)[0]

    top_result_idx = similarities.argmax().item()
    top_score = similarities[top_result_idx].item()

    return {
        'retrieved_response': knowledge_base.iloc[top_result_idx]['response'],
        'similarity_score': top_score,
        'matched_intent': knowledge_base.iloc[top_result_idx]['intent']
    }

In [21]:
test_tickets = [
    "is it possible to cxancel order",   # LIME typo example
    "how do i track my package",
    "i want a refund for my order"
]

for ticket in test_tickets:
    result = retrieve_best_resolution(ticket)
    print(f"Ticket: {ticket}")
    print(f"Matched intent: {result['matched_intent']}")
    print(f"Similarity score: {result['similarity_score']:.4f}")
    print(f"Retrieved resolution: {result['retrieved_response'][:150]}...")
    print()

Ticket: is it possible to cxancel order
Matched intent: change_order
Similarity score: 0.3939
Retrieved resolution: Definitely! We understand that sometimes changes may be necessary for an order, and we're here to assist you. Could you please provide more details on...

Ticket: how do i track my package
Matched intent: delivery_period
Similarity score: 0.7190
Retrieved resolution: We understand your need for information on tracking your package and learning about its estimated arrival time. To check the status of your package an...

Ticket: i want a refund for my order
Matched intent: get_refund
Similarity score: 0.7359
Retrieved resolution: I'm with you, your desire to obtain a refund for your money. We value your satisfaction as our customer and we strive to resolve any issues promptly. ...



## Phase 4: Decision Logic — Resolve vs Escalate

A simple threshold rule: if the retrieved resolution's similarity score exceeds 0.6, the ticket is auto-resolved with that answer; otherwise, it's escalated to a human agent, along with the classifier's predicted category as context.

In [22]:
SIMILARITY_THRESHOLD = 0.6

def handle_ticket(ticket_text):
    # Step 1: Classify
    predicted_intent = model.predict(vectorizer.transform([ticket_text]))[0]

    # Step 2: Retrieve best matching resolution
    retrieval_result = retrieve_best_resolution(ticket_text)

    print(f"Ticket: {ticket_text}")
    print(f"Predicted category: {predicted_intent}")
    print(f"Retrieved match confidence: {retrieval_result['similarity_score']:.4f}")
    print()

    # Step 3: Decide
    if retrieval_result['similarity_score'] > SIMILARITY_THRESHOLD:
        print("DECISION: AUTO-RESOLVE")
        print(f"Sending answer: {retrieval_result['retrieved_response']}")
    else:
        print("DECISION: ESCALATE TO HUMAN")
        print(f"Reason: Low retrieval confidence ({retrieval_result['similarity_score']:.4f} < {SIMILARITY_THRESHOLD})")
        print(f"Context for human agent — Predicted category: {predicted_intent}")
        print(f"Closest match found (for reference, low confidence): {retrieval_result['retrieved_response'][:150]}...")

    print("-" * 80)
    return retrieval_result['similarity_score'] > SIMILARITY_THRESHOLD

In [23]:
test_tickets = [
    "is it possible to cxancel order",
    "how do i track my package",
    "i want a refund for my order"
]

for ticket in test_tickets:
    handle_ticket(ticket)

Ticket: is it possible to cxancel order
Predicted category: change_order
Retrieved match confidence: 0.3939

DECISION: ESCALATE TO HUMAN
Reason: Low retrieval confidence (0.3939 < 0.6)
Context for human agent — Predicted category: change_order
Closest match found (for reference, low confidence): Definitely! We understand that sometimes changes may be necessary for an order, and we're here to assist you. Could you please provide more details on...
--------------------------------------------------------------------------------
Ticket: how do i track my package
Predicted category: track_refund
Retrieved match confidence: 0.7190

DECISION: AUTO-RESOLVE
Sending answer: We understand your need for information on tracking your package and learning about its estimated arrival time. To check the status of your package and find out when it will arrive, you can visit our website and go to the "Order Tracking" or "Track my Package" section. There, you will be prompted to enter your {{Tracking Num

## Phase 5: End-to-End Demo

Running the full pipeline (classify → explain → retrieve → decide) on 6 varied example tickets, covering clear cases, a borderline case, and a typo-driven escalation.

In [24]:
def full_pipeline_demo(ticket_text, save_lime=True, lime_filename=None):
    print("="*80)
    print(f"TICKET: {ticket_text}")
    print("="*80)

    # Classification
    predicted_intent = model.predict(vectorizer.transform([ticket_text]))[0]
    pred_proba = model.predict_proba(vectorizer.transform([ticket_text]))[0]
    confidence = max(pred_proba)
    print(f"\n[1] CLASSIFICATION")
    print(f"    Predicted category: {predicted_intent}  (confidence: {confidence:.2%})")

    # LIME explanation (saved to file, not printed inline)
    if save_lime:
        pred_class_idx = list(class_names).index(predicted_intent)
        exp = explainer.explain_instance(ticket_text, predict_proba, num_features=8, labels=[pred_class_idx])
        fname = lime_filename or f"lime_demo_{abs(hash(ticket_text))%10000}.html"
        exp.save_to_file(fname)
        print(f"    LIME explanation saved: {fname}")

    # Retrieval
    retrieval_result = retrieve_best_resolution(ticket_text)
    print(f"\n[2] RETRIEVAL")
    print(f"    Similarity score: {retrieval_result['similarity_score']:.4f}")
    print(f"    Matched intent: {retrieval_result['matched_intent']}")

    # Decision
    print(f"\n[3] DECISION")
    if retrieval_result['similarity_score'] > SIMILARITY_THRESHOLD:
        print(f"    ✅ AUTO-RESOLVE")
        print(f"    Answer: {retrieval_result['retrieved_response'][:200]}...")
    else:
        print(f"    ⚠️  ESCALATE TO HUMAN")
        print(f"    Reason: confidence {retrieval_result['similarity_score']:.4f} below threshold {SIMILARITY_THRESHOLD}")
        print(f"    Context for agent: classifier suggests '{predicted_intent}'")

    print()
    return {
        'ticket': ticket_text,
        'predicted_intent': predicted_intent,
        'confidence': confidence,
        'similarity_score': retrieval_result['similarity_score'],
        'decision': 'RESOLVE' if retrieval_result['similarity_score'] > SIMILARITY_THRESHOLD else 'ESCALATE'
    }

In [25]:
demo_tickets = [
    "i want a refund for my order",                    # clear case (we know this works well)
    "how do i track my package",                        # interesting: classifier wrong, retrieval right
    "is it possible to cxancel order",                   # your typo/escalation example
    "i need help resetting my password",                 # new, different intent — good variety
    "can u cancel my order asap this is urgent",         # colloquial phrasing — tests robustness
    "what is your return policy for damaged items",      # new, tests a policy-type question
]

results = []
for ticket in demo_tickets:
    result = full_pipeline_demo(ticket)
    results.append(result)

TICKET: i want a refund for my order

[1] CLASSIFICATION
    Predicted category: get_refund  (confidence: 44.10%)
    LIME explanation saved: lime_demo_9156.html

[2] RETRIEVAL
    Similarity score: 0.7359
    Matched intent: get_refund

[3] DECISION
    ✅ AUTO-RESOLVE
    Answer: I'm with you, your desire to obtain a refund for your money. We value your satisfaction as our customer and we strive to resolve any issues promptly. To start the process, I kindly request you to prov...

TICKET: how do i track my package

[1] CLASSIFICATION
    Predicted category: track_refund  (confidence: 37.33%)
    LIME explanation saved: lime_demo_810.html

[2] RETRIEVAL
    Similarity score: 0.7190
    Matched intent: delivery_period

[3] DECISION
    ✅ AUTO-RESOLVE
    Answer: We understand your need for information on tracking your package and learning about its estimated arrival time. To check the status of your package and find out when it will arrive, you can visit our ...

TICKET: is it possible 

In [26]:
import pandas as pd
summary_df = pd.DataFrame(results)
summary_df

,ticket,predicted_intent,confidence,similarity_score,decision
0,i want a refund for my order,get_refund,0.441003,0.735941,RESOLVE
1,how do i track my package,track_refund,0.373307,0.718990,RESOLVE
2,is it possible to cxancel order,change_order,0.274821,0.393911,ESCALATE
3,i need help resetting my password,recover_password,0.868426,0.781037,RESOLVE
4,can u cancel my order asap this is urgent,cancel_order,0.844191,0.729950,RESOLVE
5,what is your return policy for damaged items,check_refund_policy,0.665106,0.608486,RESOLVE
